# Synthetic hourly gas-station data generator

Usage examples for `skforecast.experimental._synthetic_data`, a generator of
realistic, messy hourly gas-station sales for training and testing forecasting
models. It reproduces bimodal weekday commutes, yearly seasonality (summer
travel, an Easter-linked travel window, Christmas), hourly weather,
random-walk fuel prices with a Monday effect, autoregressive demand momentum,
and anomalies. It lives in `skforecast.experimental` as a staging feature: the
`country` argument resolves holidays generically via the `holidays` library,
but only the default `SeasonalEffectsConfig` (Spain-tuned) has actually been
validated — other countries should pass an explicit override.

In [17]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
c:\Users\Joaquin\Documents\GitHub\skforecast
0.24.0


In [18]:
from skforecast.experimental._synthetic_data import (
    generate_gas_station_panel,
    generate_station,
    summarize,
    StationConfig,
    SeasonalEffectsConfig,
)

### 1. Generate a multi-station panel

`generate_gas_station_panel` is the main entry point. Pass an integer to
auto-configure that many stations. It returns a long-format panel (one row per
station-hour) with a frequency-aware `DatetimeIndex`.

In [19]:
panel = generate_gas_station_panel(
    start_date="2023-01-01",
    end_date="2024-12-31",
    country="ES",
    stations=5,          # mix of highway/urban, some open 24/7 and some closing overnight
    output="long",       # one row per station-hour, with a `station_id` column
    seed=42,
)
panel.head()

,station_id,liters_diesel_sold,liters_gasoline_sold,store_revenue_euros,carwash_revenue_euros,price_diesel,price_gasoline,temperature_c,precipitation_mm,is_raining,is_national_holiday,is_weekend,is_open,is_highway_station,is_anomaly
datetime,,,,,,,,,,,,,,,
2023-01-01 00:00:00,station_01,19.0,297.0,4.39,13.25,1.725,1.512,4.2,0.0,0,1,1,1,1,0
2023-01-01 01:00:00,station_01,13.0,178.0,2.93,7.36,1.725,1.512,4.1,0.0,0,1,1,1,1,0
2023-01-01 02:00:00,station_01,16.0,105.0,1.98,7.60,1.725,1.512,3.9,0.0,0,1,1,1,1,0
2023-01-01 03:00:00,station_01,21.0,109.0,1.38,5.03,1.725,1.512,3.4,0.0,0,1,1,1,1,0
2023-01-01 04:00:00,station_01,23.0,123.0,2.39,11.59,1.725,1.512,3.8,0.0,0,1,1,1,1,0


### 2. Summary report

`summarize` prints row/station counts, target ranges, the zero-when-closed
sanity check, holiday and anomaly counts, and the lag-1 sales autocorrelation
(a positive value confirms the AR momentum).

In [20]:
summarize(panel)

SYNTHETIC GAS-STATION PANEL SUMMARY
Rows                : 87,605
Date range          : 2023-01-01 00:00:00  ->  2024-12-31 00:00:00
Index frequency     : None
Stations            : 5 (station_01, station_02, station_03, station_04, station_05)
Holiday hours       : 2,640
Anomaly hours       : 263
----------------------------------------------------------------------
Targets (min / mean / max):
  liters_diesel_sold      :      0.0 /     478.33 /     4457.0
  liters_gasoline_sold    :      0.0 /     424.30 /     4673.0
  store_revenue_euros     :      0.0 /      79.16 /      626.7
  carwash_revenue_euros   :      0.0 /      14.84 /      310.4
Max sale when closed: 0.000 (should be 0)
Lag-1 autocorr sales: 0.850 (>0 means momentum)


### 3. Inspect the generated dynamics

Each row carries multi-product targets (diesel/gasoline liters, store and
car-wash revenue) plus exogenous features (prices, hourly weather, holiday /
weekend / open flags, and an anomaly marker).

In [21]:
# Full column set: multi-product targets + exogenous features
print(panel.columns.tolist())

# Diurnal pattern for one station: coldest before dawn, hottest mid-afternoon,
# and the bimodal commute shape of gasoline sales
station = panel[panel["station_id"] == "station_01"]
station.groupby(station.index.hour)[["temperature_c", "liters_gasoline_sold"]].mean()

['station_id', 'liters_diesel_sold', 'liters_gasoline_sold', 'store_revenue_euros', 'carwash_revenue_euros', 'price_diesel', 'price_gasoline', 'temperature_c', 'precipitation_mm', 'is_raining', 'is_national_holiday', 'is_weekend', 'is_open', 'is_highway_station', 'is_anomaly']


,temperature_c,liters_gasoline_sold
datetime,,
0,12.473051,92.813953
1,11.619726,55.173973
2,11.091370,37.549315
3,10.915616,37.757534
4,11.056301,57.875342
5,11.609452,98.897260
6,12.474795,379.530137
7,13.651644,820.182192
8,14.948082,781.693151


### 4. Custom station configurations

Instead of an integer, pass an explicit list of `StationConfig` instances to
control each station's type (highway vs urban), opening hours, base demand,
and base prices. Holidays are resolved internally for the requested date
range.

In [22]:
custom_configs = [
    StationConfig(
        station_id="highway_north",
        is_highway=True,
        closes_at_night=False,       # open 24/7
        base_diesel_per_hr=700.0,
        base_gasoline_per_hr=500.0,
        base_price_diesel=1.70,
        base_price_gasoline=1.60,
    ),
    StationConfig(
        station_id="urban_center",
        is_highway=False,
        closes_at_night=True,        # closed 23:00-05:59
        base_diesel_per_hr=450.0,
        base_gasoline_per_hr=350.0,
        base_price_diesel=1.60,
        base_price_gasoline=1.50,
    ),
]

custom = generate_gas_station_panel(
    start_date="2024-01-01",
    end_date="2024-12-31",
    stations=custom_configs,
    seed=7,
)
custom.groupby("station_id")[
    ["liters_diesel_sold", "liters_gasoline_sold", "is_open"]
].mean()

C:\Users\Joaquin\AppData\Local\Temp\ipykernel_27348\3196391697.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  custom.groupby("station_id")[


,liters_diesel_sold,liters_gasoline_sold,is_open
station_id,,,
highway_north,555.460336,568.098847,1.000000
urban_center,295.920100,293.947837,0.708252


### 5. Wide-format output

Pass `output="wide"` to get a single `DatetimeIndex` with columns suffixed by
station id (convenient for correlation analysis or plotting).

In [23]:
wide = generate_gas_station_panel(
    start_date="2024-01-01",
    end_date="2024-03-31",
    stations=2,
    output="wide",
    seed=42,
)
wide.iloc[:3, :6]

,liters_diesel_sold_station_01,liters_gasoline_sold_station_01,store_revenue_euros_station_01,carwash_revenue_euros_station_01,price_diesel_station_01,price_gasoline_station_01
datetime,,,,,,
2024-01-01 00:00:00,40.0,109.0,2.86,6.60,1.711,1.5
2024-01-01 01:00:00,25.0,63.0,1.51,3.51,1.711,1.5
2024-01-01 02:00:00,57.0,56.0,1.48,2.31,1.711,1.5


### 6. Explore the generated series with Plotly

Interactive plots to sanity-check the temporal structure of the panel from
section 1: the intraday commute shape, the weekly and yearly seasonality, and
the price-to-volume relationship. Requires `plotly` (`pip install plotly`).

In [24]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Explore a single station from the panel generated in section 1
sid = "station_01"
station = panel[panel["station_id"] == sid].copy()
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print(f"Exploring {sid}: {station.shape[0]:,} hourly rows")

Exploring station_01: 17,521 hourly rows


**Hourly detail.** Zooming into a two-week window shows the intraday shape the
generator produces: twin weekday commute peaks around 07-09 and 17-19, a single
broad midday hump at weekends, and near-zero overnight demand.

In [25]:
# Zoom into a two-week window to see the hourly shape: twin weekday commute
# peaks (~07-09 and ~17-19), a single broad weekend hump, and near-zero nights.
window = station.loc["2024-03-04":"2024-03-17"]

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=window.index, y=window["liters_gasoline_sold"],
    name="Gasoline (L)", line=dict(color="#e45756", width=1.5),
))
fig1.add_trace(go.Scatter(
    x=window.index, y=window["liters_diesel_sold"],
    name="Diesel (L)", line=dict(color="#4c78a8", width=1.5),
))
fig1.update_layout(
    title=f"{sid}: hourly sales over two weeks (bimodal weekday commute, wide weekend peak)",
    xaxis_title="datetime", yaxis_title="liters sold",
    template="plotly_white", hovermode="x unified", height=420,
)
fig1.show()

**Weekly x daily structure.** A heatmap of mean gasoline sales across the
hour-of-day / day-of-week grid makes the bimodal weekday commute and the wider
single weekend peak visible at a glance.

In [26]:
# Mean gasoline sales by hour of day (rows) and day of week (columns). Weekdays
# show the twin commute bands; weekends collapse into a single midday block.
heat = (
    station.groupby([station.index.dayofweek, station.index.hour])["liters_gasoline_sold"]
    .mean()
    .unstack(level=0)
)

fig2 = go.Figure(go.Heatmap(
    z=heat.values, x=day_names, y=heat.index,
    colorscale="Turbo", colorbar=dict(title="mean L"),
))
fig2.update_layout(
    title=f"{sid}: mean gasoline sales by hour of day and day of week",
    xaxis_title="day of week", yaxis_title="hour of day",
    template="plotly_white", height=520,
)
fig2.show()

**Yearly seasonality.** Aggregating to daily totals exposes the Spanish
calendar: the summer "Operacion Salida" travel surge, the Semana Santa window,
and Christmas. National holidays are overlaid as markers.

In [27]:
# Daily totals across the full range reveal the yearly seasonality: the summer
# "Operacion Salida" surge (July/August), Semana Santa, and the Christmas window.
daily = station.resample("D").agg(
    diesel=("liters_diesel_sold", "sum"),
    gasoline=("liters_gasoline_sold", "sum"),
    is_holiday=("is_national_holiday", "max"),
)
holidays = daily.index[daily["is_holiday"] == 1]

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=daily.index, y=daily["diesel"], name="Diesel (L/day)",
    line=dict(color="#4c78a8", width=1),
))
fig3.add_trace(go.Scatter(
    x=daily.index, y=daily["gasoline"], name="Gasoline (L/day)",
    line=dict(color="#e45756", width=1),
))
fig3.add_trace(go.Scatter(
    x=holidays, y=daily.loc[holidays, "gasoline"], mode="markers",
    name="National holiday",
    marker=dict(color="#f2b701", size=6, symbol="diamond"),
))
fig3.update_layout(
    title=f"{sid}: daily total sales (summer Operacion Salida, Semana Santa, Christmas)",
    xaxis_title="date", yaxis_title="liters/day",
    template="plotly_white", hovermode="x unified", height=440,
)
fig3.show()

**Exogenous driver: price vs volume.** Fuel prices follow a persistent random
walk (not IID noise), and demand carries a negative price elasticity, so
sustained price rises visibly depress daily volume.

In [28]:
# Exogenous price vs volume: as the diesel price random-walks up, daily volume
# tends to fall (negative elasticity). Dual y-axis to overlay the two scales.
daily_px = station.resample("D").agg(
    diesel=("liters_diesel_sold", "sum"),
    price=("price_diesel", "mean"),
)

fig4 = make_subplots(specs=[[{"secondary_y": True}]])
fig4.add_trace(
    go.Scatter(
        x=daily_px.index, y=daily_px["diesel"], name="Diesel (L/day)",
        line=dict(color="#4c78a8", width=1),
    ),
    secondary_y=False,
)
fig4.add_trace(
    go.Scatter(
        x=daily_px.index, y=daily_px["price"], name="Diesel price (EUR/L)",
        line=dict(color="#54a24b", width=1.5),
    ),
    secondary_y=True,
)
fig4.update_layout(
    title=f"{sid}: fuel price vs volume (price rises depress demand)",
    template="plotly_white", hovermode="x unified", height=440,
)
fig4.update_yaxes(title_text="liters/day", secondary_y=False)
fig4.update_yaxes(title_text="EUR/L", secondary_y=True)

corr = daily_px["diesel"].corr(daily_px["price"])
print(f"Daily price/volume correlation: {corr:.3f} (negative = elasticity)")
fig4.show()

Daily price/volume correlation: -0.163 (negative = elasticity)


### 7. Training a Global Forecasting Model

A powerful approach for handling multiple related time series is to train a single **Global Model**. Instead of training an individual model for each gas station and each fuel type, we can combine all series into a single dataset. Algorithms like `LightGBM` can learn shared patterns across all series (cross-learning).

**Data Restructuring Strategy:**
To achieve this, we need to treat each combination of `station_id` and `fuel type` as a unique, independent series.

1. **Melt the DataFrame:** We transform the data from "wide" (gasoline and diesel in separate columns) to "long" format (a single `liters_sold` column with a new `product` column indicating the fuel type). This gives us our new `series_id`.
2. **Categorical Features:** We convert `station_id` and `product` to Pandas `category` types. LightGBM natively supports categorical variables without needing One-Hot Encoding, allowing it to efficiently learn the baseline differences between stations and products.
3. **Cross-Price Elasticity:** By melting the data, the exogenous dictionaries will duplicate the price information. This is exactly what we want! For example, the `station_01_gasoline` series will have access to the `price_diesel` exogenous variable, allowing the model to learn complex dynamics like cross-price elasticity (e.g., if diesel is expensive, do gasoline sales change?).

For more information on handling multiple series in `skforecast`, check the [Independent multi-time series forecasting documentation](https://skforecast.org/latest/user_guides/independent-multi-time-series-forecasting.html).

In [29]:
from lightgbm import LGBMRegressor
from skforecast.recursive import ForecasterRecursiveMultiSeries
from skforecast.preprocessing import reshape_series_long_to_dict, reshape_exog_long_to_dict

# 1. Split panel into train and future (test) to have future exog data for predict
# The panel ends at 2024-12-31 00:00:00, so we use the last 12 hours as the future set.
train_panel = panel[panel.index < "2024-12-30 13:00:00"].reset_index()
future_panel = panel[panel.index >= "2024-12-30 13:00:00"].reset_index()

# 2. Melt data to treat each product at each station as an independent series
exog_cols = [
    "price_gasoline", "price_diesel", "temperature_c", "precipitation_mm", 
    "is_raining", "is_national_holiday", "is_weekend", 
    "is_open", "is_highway_station", "is_anomaly"
]

def prep_melted_panel(df):
    df_melted = df.melt(
        id_vars=["datetime", "station_id"] + exog_cols,
        value_vars=["liters_gasoline_sold", "liters_diesel_sold"],
        var_name="product",
        value_name="liters_sold"
    )
    df_melted["product"] = df_melted["product"].str.replace("liters_", "").str.replace("_sold", "")
    df_melted["series_id"] = df_melted["station_id"].astype(str) + "_" + df_melted["product"]
    df_melted["station_id"] = df_melted["station_id"].astype("category")
    df_melted["product"] = df_melted["product"].astype("category")
    return df_melted

train_melted = prep_melted_panel(train_panel)
future_melted = prep_melted_panel(future_panel)

# Update exog_cols to include the new categorical features
exog_cols_new = exog_cols + ["station_id", "product"]

# 3. Reshape series and exogenous variables into dictionaries
series_dict = reshape_series_long_to_dict(
    data      = train_melted,
    series_id = "series_id",
    index     = "datetime",
    values    = "liters_sold",
    freq      = "h"
)

exog_dict = reshape_exog_long_to_dict(
    data      = train_melted[["series_id", "datetime"] + exog_cols_new],
    series_id = "series_id",
    index     = "datetime",
    freq      = "h"
)

# 4. Reshape the exogenous variables for prediction
exog_dict_future = reshape_exog_long_to_dict(
    data      = future_melted[["series_id", "datetime"] + exog_cols_new],
    series_id = "series_id",
    index     = "datetime",
    freq      = "h"
)

In [30]:
forecaster = ForecasterRecursiveMultiSeries(
    estimator=LGBMRegressor(n_estimators=100, random_state=123, verbose=-1),
    lags=24,
)

forecaster.fit(series=series_dict, exog=exog_dict)
predictions = forecaster.predict(steps=12, exog=exog_dict_future)

In [31]:
fig = go.Figure()
series_keys = list(series_dict.keys())

# Handle long or wide format output seamlessly
preds_wide = predictions.pivot(columns="level", values="pred") if "level" in predictions.columns else predictions

for i, sid in enumerate(series_keys):
    actuals = future_melted[future_melted["series_id"] == sid]
    preds = preds_wide[sid]
    
    visible = True if i == 0 else False
    
    fig.add_trace(go.Scatter(
        x=actuals["datetime"], y=actuals["liters_sold"],
        name=f"{sid} (Actual)", line=dict(color="#4c78a8", width=1.5),
        visible=visible
    ))
    
    fig.add_trace(go.Scatter(
        x=preds.index, y=preds,
        name=f"{sid} (Predicted)", line=dict(color="#e45756", width=1.5, dash="dash"),
        visible=visible
    ))

buttons = []
for i, sid in enumerate(series_keys):
    visibility = [False] * (len(series_keys) * 2)
    visibility[i * 2] = True
    visibility[i * 2 + 1] = True
    
    button = dict(
        label=sid,
        method="update",
        args=[{"visible": visibility},
              {"title": f"{sid}: Predicted vs Actual (Next 12 Hours)"}]
    )
    buttons.append(button)

fig.update_layout(
    title=f"{series_keys[0]}: Predicted vs Actual (Next 12 Hours)",
    xaxis_title="datetime", yaxis_title="liters sold",
    template="plotly_white", hovermode="x unified", height=440,
    updatemenus=[dict(
        active=0, buttons=buttons, 
        x=1.0, xanchor="right", y=1.15, yanchor="top"
    )]
)

fig.show()